In [1]:
import pandas as pd
import numpy as np
import math
from collections import Counter
from scipy.spatial.distance import jensenshannon
def generate_llm_samples(corpus, dataset_name, model_version):
    
    total = len(corpus)
    np.random.seed(42)
    
    sample_1k_size = min(1000, total)
    sample_10k_size = min(10000, total)
    
    indices_1k = np.random.choice(total, sample_1k_size, replace=False)
    indices_10k = np.random.choice(total, sample_10k_size, replace=False)
    
    file_1k = f"{dataset_name}_{model_version}_1K.txt"
    file_10k = f"{dataset_name}_{model_version}_10K.txt"
    
    with open(file_1k, "w", encoding="utf-8") as f:
        for i in indices_1k:
            f.write(corpus[i] + "\n")
    
    with open(file_10k, "w", encoding="utf-8") as f:
        for i in indices_10k:
            f.write(corpus[i] + "\n")
    
    print(f"LLM sample files generated for {dataset_name} ({model_version})")
    print(" -", file_1k)
    print(" -", file_10k)
# GENERIC SEMANTIC MODEL (PARAMETRIC VERSION)
def semantic_model(file_path, dataset_name, feature_version="SFE-5"):

    print(f"SEMANTIC ANALYSIS – {feature_version} – {dataset_name}")
   
    df = pd.read_csv(file_path)
    X = df.drop("Label", axis=1)
    
    print("Dataset shape:", X.shape)

    # SELECT FEATURES

    base_features = {
        "bytes": "Flow Bytes/s",
        "packets": "Flow Packets/s",
        "duration": "Flow Duration",
        "pkt_std": "Packet Length Std",
        "ratio": "Down/Up Ratio"
    }
    
    extended_features = {
        "pkt_mean": "Packet Length Mean",
        "iat_mean": "Flow IAT Mean",
        "active_mean": "Active Mean"
    }
    
    selected = base_features.copy()
    
    if feature_version == "SFE-8":
        selected.update(extended_features)
        print("Using extended 8-feature semantic model.")
    else:
        print("Using baseline 5-feature semantic model.")
    # QUINTILES
    quintiles = {}
    for key, col in selected.items():
        quintiles[key] = X[col].quantile([0.2, 0.4, 0.6, 0.8]).values
    
    print("Quintile thresholds computed.")
    # CATEGORIZATION FUNCTION
    def categorize(value, thresholds):
        if value < thresholds[0]:
            return "very low"
        elif value < thresholds[1]:
            return "low"
        elif value < thresholds[2]:
            return "moderate"
        elif value < thresholds[3]:
            return "high"
        else:
            return "very high"
    # SEMANTIC ENCODER
    def semantic_encode(row):
        
        values = {}
        for key, col in selected.items():
            values[key] = categorize(row[col], quintiles[key])
        
        protocol = "TCP" if row["Protocol"] == 6 else "UDP"
        
        throughput = values["bytes"]
        packet_rate = values["packets"]
        duration = values["duration"]
        variability = values["pkt_std"]
        directionality = values["ratio"]
        
        if throughput in ["high", "very high"] and packet_rate in ["high", "very high"]:
            pattern = "burst-like transmission pattern"
        elif throughput == "very low" and duration in ["high", "very high"]:
            pattern = "persistent low-volume communication"
        elif variability in ["high", "very high"] and duration in ["very low", "low"]:
            pattern = "irregular short burst behavior"
        else:
            pattern = "stable communication behavior"
        
        description = (
            f"A {duration} {protocol}-based flow with {throughput} throughput, "
            f"{packet_rate} packet rate, {directionality} directional balance, "
            f"{variability} packet size variability"
        )
        
        if feature_version == "SFE-8":
            description += (
                f", {values['pkt_mean']} average packet size, "
                f"{values['iat_mean']} inter-arrival timing, "
                f"{values['active_mean']} active phase duration"
            )
        
        description += f", exhibiting a {pattern}."
        
        return description
    
    print("Generating full semantic corpus...")
    corpus = [semantic_encode(X.iloc[i]) for i in range(len(X))]
    
    total = len(corpus)
    counts = Counter(corpus)
    unique_count = len(counts)
    uniqueness_ratio = unique_count / total
    
    entropy = 0
    for count in counts.values():
        p = count / total
        entropy -= p * math.log2(p)
    
    print("\n----- RESULTS -----")
    print("Total flows:", total)
    print("Unique semantic patterns:", unique_count)
    print("Uniqueness ratio:", round(uniqueness_ratio, 6))
    print("Shannon entropy:", round(entropy, 6))
    
    generate_llm_samples(corpus, dataset_name, feature_version)
    
    probs = {k: v / total for k, v in counts.items()}
    
    return probs, entropy
# RUN SFE-5
benign_probs_5, benign_entropy_5 = semantic_model(
    "Benign_All_Flow_Clean_FINAL.csv",
    "BENIGN",
    feature_version="SFE-5"
)

ddos_probs_5, ddos_entropy_5 = semantic_model(
    "DDoS_Cleaned.csv",
    "DDOS",
    feature_version="SFE-5"
)

# RUN SFE-8
benign_probs_8, benign_entropy_8 = semantic_model(
    "Benign_All_Flow_Clean_FINAL.csv",
    "BENIGN",
    feature_version="SFE-8"
)

ddos_probs_8, ddos_entropy_8 = semantic_model(
    "DDoS_Cleaned.csv",
    "DDOS",
    feature_version="SFE-8"
)

# JENSEN-SHANNON DIVERGENCE
def compute_jsd(p_dict, q_dict):
    all_patterns = set(p_dict.keys()).union(set(q_dict.keys()))
    P = np.array([p_dict.get(p, 0) for p in all_patterns])
    Q = np.array([q_dict.get(p, 0) for p in all_patterns])
    return jensenshannon(P, Q, base=2) ** 2


jsd_5 = compute_jsd(benign_probs_5, ddos_probs_5)
jsd_8 = compute_jsd(benign_probs_8, ddos_probs_8)


print("\n==================================================")
print("INTER-CLASS INFORMATIONAL ANALYSIS")
print("==================================================")
print("SFE-5 Benign Entropy:", round(benign_entropy_5, 6))
print("SFE-5 DDoS Entropy:", round(ddos_entropy_5, 6))
print("SFE-5 Jensen-Shannon Divergence:", round(jsd_5, 6))
print("--------------------------------------------------")
print("SFE-8 Benign Entropy:", round(benign_entropy_8, 6))
print("SFE-8 DDoS Entropy:", round(ddos_entropy_8, 6))
print("SFE-8 Jensen-Shannon Divergence:", round(jsd_8, 6))
print("==================================================\n")



SEMANTIC ANALYSIS – SFE-5 – BENIGN
Dataset shape: (398198, 70)
Using baseline 5-feature semantic model.
Quintile thresholds computed.
Generating full semantic corpus...

----- RESULTS -----
Total flows: 398198
Unique semantic patterns: 506
Uniqueness ratio: 0.001271
Shannon entropy: 6.517957
LLM sample files generated for BENIGN (SFE-5)
 - BENIGN_SFE-5_1K.txt
 - BENIGN_SFE-5_10K.txt
SEMANTIC ANALYSIS – SFE-5 – DDOS
Dataset shape: (2954569, 70)
Using baseline 5-feature semantic model.
Quintile thresholds computed.
Generating full semantic corpus...

----- RESULTS -----
Total flows: 2954569
Unique semantic patterns: 263
Uniqueness ratio: 8.9e-05
Shannon entropy: 4.539669
LLM sample files generated for DDOS (SFE-5)
 - DDOS_SFE-5_1K.txt
 - DDOS_SFE-5_10K.txt
SEMANTIC ANALYSIS – SFE-8 – BENIGN
Dataset shape: (398198, 70)
Using extended 8-feature semantic model.
Quintile thresholds computed.
Generating full semantic corpus...

----- RESULTS -----
Total flows: 398198
Unique semantic patterns: